# 🕵️ The Hidden Layer — Micro Mission *(Solutions)*

```
╔══════════════════════════════════════════════════════════╗
║         C L A S S I F I E D   B R I E F I N G          ║
║                  AGENT LAMBDA — MICRO                    ║
╚══════════════════════════════════════════════════════════╝
```

> *"The base is small. The mission is simple. But the agent is you — or rather, it's an LLM you put in charge."*

---

## What is this?

You are going to build an **AI agent** — a program that perceives its environment, thinks using a large language model, and acts through a set of tools.

The agent controls a spy operative on a **3×3 grid**. Every turn, the agent:
1. **Perceives** — receives a scan of its surroundings
2. **Thinks** — calls an LLM to decide what to do
3. **Acts** — executes one tool

Your job is to implement the **think step**: write a `think_llm()` function that calls Gemini and returns a valid tool call.

---

## The Mission

| What | Where | Reward |
|------|-------|--------|
| 📁 Dossier cache | Corner of the base | 1 dossier |
| 🤝 USB delivery | Talk to Dr. Vapnik → deliver to Agent Dropout | 1 dossier |
| 🤖 Destroy robot | Get Flamethrower from jungle → move into robot | 1 dossier |
| **Total needed** | | **3 dossiers** |

**30 turns. 3×3 grid. No walls. Go.**


In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────────────
NOTEBOOK_VERSION = "v1.3"
print(f"Notebook version: {NOTEBOOK_VERSION}")

import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if os.path.exists("coding-exercises"):
        !cd coding-exercises && git pull
    else:
        !git clone -b no-bfs https://github.com/eth-bmai-fs26/coding-exercises.git
    os.chdir("coding-exercises/agentic_ai_spy")

sys.path.insert(0, ".")

!pip install google-genai --quiet
print("Setup complete ✓")

In [ ]:
# ── Build the micro mission world ──────────────────────────────────────────────────

import os
import base64
from IPython.display import display as ipy_display, HTML as IPyHTML

from hidden_layer.game_world import CellType, Cell, NPC, GameWorld
from hidden_layer.operative import Operative
from hidden_layer.tools import GameTools, ToolResult
from hidden_layer.oracle import stub_oracle, llm_oracle
from hidden_layer.agent import TOOLS_DESCRIPTION, parse_tool_call, run_agent
from hidden_layer.display import display_turn, display_final

# Assets are in ./assets/ relative to the agentic_ai_spy working directory
_ASSETS = "assets"

NPC_IMAGES = {
    "dr_vapnik": os.path.join(_ASSETS, "dr_vapnik.jpg"),
    "dropout":   os.path.join(_ASSETS, "agent_dropout.jpg"),
}

def _npc_portrait_html(npc_id: str) -> str:
    """Return an HTML img tag with the NPC portrait embedded as base64, or empty string."""
    path = NPC_IMAGES.get(npc_id, "")
    if not path or not os.path.exists(path):
        return ""
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    ext = path.rsplit(".", 1)[-1].lower()
    return (
        f'<img src="data:image/{ext};base64,{b64}" '
        f'style="width:120px;height:120px;object-fit:cover;border-radius:8px;'
        f'float:left;margin:0 10px 4px 0;border:2px solid #333;" />'
    )


# ── Micro NPCs ───────────────────────────────────────────────────────────────────

MICRO_NPC_CATALOG = {
    "dr_vapnik": NPC(
        name="Dr. Vapnik",
        personality="Helpful old scientist. Gets to the point quickly.",
        knowledge=[
            "He has a USB Drive that must reach Agent Dropout at position (1, 1) in the center of the base.",
            "Delivering the USB Drive pays 1 dossier.",
        ],
        style="Speaks in brief statistical metaphors but always delivers clear information.",
        greeting="Ah, agent. I have a job for you. Ask me about it.",
    ),
    "dropout": NPC(
        name="Agent Dropout",
        personality="Bitter but helpful burned spy. Very direct.",
        knowledge=[
            "She receives USB Drives. Will pay 1 dossier for one.",
            "The Cryo-Sentinel robot at position (2, 2) freezes intruders.",
            "A Flamethrower can destroy the Cryo-Sentinel. Worth 1 dossier.",
            "There is a Flamethrower hidden in the jungle at the northwest corner, position (0, 0).",
        ],
        style="Direct and blunt. No riddles, no metaphors. Tells you exactly what you need.",
        greeting="You again? Fine. What do you need?",
    ),
}


# ── Micro stub oracle ────────────────────────────────────────────────────────────

def micro_stub_oracle(npc, message, operative):
    """Simple keyword-matched responses for micro mission."""
    q = message.lower()
    name = npc.name

    if name == "Dr. Vapnik":
        if any(kw in q for kw in ["usb", "drive", "deliver", "job", "work", "task",
                                    "errand", "help", "mission", "what", "how", "tell"]):
            return ("I have a USB Drive with critical data. Deliver it to Agent Dropout "
                    "at position (1, 1) \u2014 the center of the base. 1 dossier for the job.")
        return "I have a job for you, agent. Ask me about a delivery or a job."

    if name == "Agent Dropout":
        if any(kw in q for kw in ["usb", "drive", "deliver"]) and operative.has_item("USB Drive"):
            return "The USB drive! Good. Here's your dossier."
        if any(kw in q for kw in ["robot", "cryo", "sentinel", "fire", "flame", "weapon",
                                    "help", "mission", "what", "how", "tell", "job", "task"]):
            return ("There's a Cryo-Sentinel robot at position (2, 2). It freezes everything. "
                    "Move into it with a Flamethrower to destroy it \u2014 worth 1 dossier. "
                    "There's a Flamethrower in the jungle at the northwest corner, position (0, 0).")
        return "Ask me about the robot or if you have something to deliver."

    return f"{name} says nothing useful."


# ── Micro Game World ─────────────────────────────────────────────────────────────

class MicroGameWorld(GameWorld):
    """3x3 micro mission grid — no walls, open terrain."""

    ROWS = 3
    COLS = 3

    def __init__(self):
        self.grid = []
        self.usb_drive_picked_up = False
        self.usb_drive_delivered = False
        self.microfilm_picked_up = False
        self.microfilm_delivered = False
        self.codebook_picked_up = False
        self.codebook_delivered = False
        self.hard_drive_traded = False
        self.medical_supplies_delivered = False
        self.virus_code_received = False
        self.cryo_sentinel_alive = True
        self.evil_ai_robot_alive = False
        self.build_map()

    def build_map(self):
        self.grid = [
            [Cell(CellType.OPEN) for _ in range(self.COLS)]
            for _ in range(self.ROWS)
        ]

        # Row 0: 🌴 · 📁
        self._set(0, 0, Cell(CellType.JUNGLE, items=["Flamethrower"],
            description="Dense jungle at the northwest corner. A Flamethrower is stashed under the roots!"))
        self._set(0, 2, Cell(CellType.CACHE, items=["dossier_1"],
            description="A filing cabinet left unlocked. Classified documents inside!"))

        # Row 1: · 🕵️ ·
        self._set(1, 1, Cell(CellType.INFORMANT, npc_id="dropout",
            description="A camouflaged hideout. A woman sharpens a knife."))

        # Row 2: 🟢 🕵️ 🤖
        self._set(2, 0, Cell(CellType.OPEN,
            description="The south shore. This is where you came ashore."))
        self._set(2, 1, Cell(CellType.INFORMANT, npc_id="dr_vapnik",
            description="A weathered shack. An old man scribbles equations in the dirt."))
        self._set(2, 2, Cell(CellType.ROBOT, robot_name="Cryo-Sentinel",
            description="A freezing corridor. A hulking robot blocks the path, frost pouring from its vents."))

    def _set(self, row, col, cell):
        self.grid[row][col] = cell


# ── Micro tools (patched for micro NPCs) ───────────────────────────────────────

class MicroGameTools(GameTools):
    """Game tools patched for micro mission NPCs."""

    def talk(self, message=""):
        row, col = self.operative.position
        cell = self.world.get_cell(row, col)

        if cell.cell_type == CellType.INFORMANT and cell.npc_id:
            npc = MICRO_NPC_CATALOG.get(cell.npc_id)
            if not npc:
                return ToolResult(False, "Unknown informant.")
            if self._oracle_fn is None:
                return ToolResult(False, "No oracle function set.")

            response = self._oracle_fn(npc, message, self.operative)

            # Dr. Vapnik gives USB Drive
            if cell.npc_id == "dr_vapnik" and not self.world.usb_drive_picked_up:
                if any(kw in message.lower() for kw in ["usb", "drive", "job", "work", "task", "delivery", "errand"]):
                    self.operative.add_item("USB Drive")
                    self.world.usb_drive_picked_up = True
                    response += "\n[Dr. Vapnik hands you a USB Drive.]"

            # Dropout receives USB Drive
            if cell.npc_id == "dropout" and self.operative.has_item("USB Drive") and not self.world.usb_drive_delivered:
                if any(kw in message.lower() for kw in ["usb", "drive", "deliver", "data"]):
                    self.operative.remove_item("USB Drive")
                    self.operative.add_dossiers(1)
                    self.world.usb_drive_delivered = True
                    response += "\n[You delivered the USB Drive! +1 dossier.]"

            self.operative.journal.append(
                f"Talked to {npc.name}: '{message}' \u2192 '{response[:300]}'"
            )

            # Embed portrait inline with the speech
            portrait = _npc_portrait_html(cell.npc_id)
            full_message = f'{portrait}<b>{npc.name} says:</b> {response}<div style="clear:both"></div>'
            return ToolResult(True, full_message)

        return ToolResult(False, "There is no one to talk to here.")


# ── Micro mission briefing ───────────────────────────────────────────────────────

MICRO_MISSION_BRIEFING = """CLASSIFIED \u2014 MICRO MISSION BRIEFING \u2014 AGENT LAMBDA

OBJECTIVE: Extract 3 classified dossiers from a tiny OVERFIT outpost (3x3 grid).

THE BASE:
- Dossier caches (\U0001f4c1): Use collect() to grab them. Worth 1 dossier each.
- Jungle (\U0001f334): May contain useful items. Use collect() to search.
- Informants (\U0001f575\ufe0f): Talk using talk(). Ask about "jobs" or "deliveries" to get
  quest items. If you have an item to deliver, mention it by name.
- Robot (\U0001f916): Move into it with the correct weapon to destroy it (+1 dossier).
  Moving in WITHOUT the weapon deals 1 damage and bounces you back.

HOW TO EARN DOSSIERS:
1. Collect the dossier cache (1 dossier)
2. Complete the delivery errand (1 dossier) \u2014 ask informants about "jobs"
3. Destroy the robot by moving into it with the right weapon (1 dossier)

KEY TACTICS:
- Talk to EVERY informant. They tell you exactly what to do.
- When told to deliver something, go to the destination and mention the item.
- When told about a weapon location, go get it, then move into the robot cell.
- Don't revisit cells you've already collected from.
"""


# ── Micro mission runner ─────────────────────────────────────────────────────────

def create_micro_game():
    world = MicroGameWorld()
    operative = Operative(position=(2, 0), visited={(2, 0)})
    operative.WIN_DOSSIERS = 3
    tools = MicroGameTools(operative, world)
    return operative, world, tools


def play_micro_mission(think_fn, oracle_fn=None, max_turns=30, show_display=True, delay=0.3):
    operative, world, tools = create_micro_game()
    tools.set_oracle(oracle_fn or micro_stub_oracle)

    disp = None
    if show_display:
        disp = lambda w, o, t, a, r, s="": display_turn(w, o, t, a, r, scan_result=s, delay=delay)

    import hidden_layer.agent as _agent_mod
    _orig_briefing = _agent_mod.MISSION_BRIEFING
    _agent_mod.MISSION_BRIEFING = MICRO_MISSION_BRIEFING
    try:
        result = run_agent(think_fn, operative, world, tools, max_turns, display_fn=disp)
    finally:
        _agent_mod.MISSION_BRIEFING = _orig_briefing

    if show_display:
        display_final(operative, result["turns"])

    return result


print("Micro mission loaded! \u2713")
print("Map: 3\u00d73 | Goal: 3 dossiers | Max turns: 30")


In [ ]:
# ── Show the map ─────────────────────────────────────────────────────────────────
operative, world, tools = create_micro_game()

lines = ["    " + "  ".join(f" {c} " for c in range(world.COLS))]
lines.append("   " + "----" * world.COLS + "-")
for row in range(world.ROWS):
    row_str = f"{row}  |"
    for col in range(world.COLS):
        cell = world.get_cell(row, col)
        if (row, col) == operative.position:
            row_str += " \U0001f7e2 |"
        else:
            emoji = cell.cell_type.emoji
            row_str += f" {emoji}  |" if len(emoji) == 1 else f" {emoji} |"
    lines.append(row_str)
    lines.append("   " + "----" * world.COLS + "-")
print("\n".join(lines))

---
## Play the Game Yourself!

Try the micro mission manually before looking at the AI solution. Move around, talk to informants, collect items, and see if you can grab all 3 dossiers in 30 turns!

In [ ]:
from hidden_layer.interactive import InteractiveGame

game = InteractiveGame(
    oracle_fn=micro_stub_oracle,
    game_factory=create_micro_game,
    max_turns=30,
    goal_dossiers=3,
    title="MICRO MISSION",
)
game.play()

In [ ]:
# ── Try some tools ───────────────────────────────────────────────────────────────
tools.set_oracle(micro_stub_oracle)

print("=== Scan from (4,0) ===")
print(tools.execute("scan", {}).message)

print("\n=== Move to Dr. Vapnik (4,1) ===")
print(tools.execute("move", {"direction": "east"}).message)

print("\n=== Ask Vapnik for a job ===")
print(tools.execute("talk", {"message": "Do you have a job for me?"}).message)
print(f"Inventory: {operative.inventory}")

---
## API Key Setup

In [ ]:
import os
from google import genai

try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except (ImportError, Exception):
    api_key = os.environ.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)
print("Gemini client ready!")

---
## TODO: Implement think_llm

The LLM decides the operative's action each turn. It receives the game state via `history` (which includes the mission briefing on turn 0).

```python
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=user_message,
    config=genai.types.GenerateContentConfig(
        system_instruction=system_message,
        max_output_tokens=500,
    ),
)
return response.text
```

In [ ]:
SYSTEM_PROMPT = """You are an AI agent controlling a spy on a 3x3 base. Collect 3 dossiers in 30 turns.

{tools}

RULES:
- The grid is 3x3. Rows and columns are numbered 0-2.
- Each turn you receive a scan showing your position and adjacent cells.
- Talk to informants to get quests. Ask about "jobs", "deliveries", or "what should I do?".
- If you have an item to deliver, mention it by name when talking to the recipient.
- Collect items from jungle cells and dossier caches using collect().
- Move into a robot cell with the correct weapon to destroy it.

WHEN LOST OR UNCERTAIN:
- Find the nearest informant and talk to them.
- Ask Dr. Vapnik: "Do you have a job for me?" or "What should I do?"
- Ask Agent Dropout: "What do you know about the robot?" or "What should I do next?"
- Informants always know what the next step is — ask them!

Respond with EXACTLY one tool call, on the last line, in this format:
TOOL: move(direction="north")
TOOL: talk(message="Do you have a job for me?")
TOOL: collect()"""


def think_llm(operative, world, history, client):
    """Minimal think function — build a prompt from history and call the LLM."""
    # Format history as a simple transcript
    transcript = []
    for h in history[-20:]:
        if h["role"] == "observation":
            transcript.append(f"SCAN: {h['content']}")
        elif h["role"] == "action":
            transcript.append(f"ACTION: {h['content']}")
        elif h["role"] == "result":
            transcript.append(f"RESULT: {h['content']}")

    user_msg = "\n".join(transcript) + "\n\nWhat do you do next?"

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=user_msg,
        config=genai.types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT.format(tools=TOOLS_DESCRIPTION),
            max_output_tokens=200,
            temperature=0.3,
        ),
    )
    return response.text


In [ ]:
# ── Run the micro mission ────────────────────────────────────────────────────────
oracle_fn = lambda npc, q, o: llm_oracle(npc, q, o, client)

result = play_micro_mission(
    think_fn=lambda operative, world, history: think_llm(operative, world, history, client),
    oracle_fn=oracle_fn,
    max_turns=30,
    delay=0.3,
)
print(f"\nResult: {'Mission Complete!' if result['won'] else 'Mission Failed...'}")
print(f"Dossiers: {result['dossiers']} | Health: {result['health']} | Turns: {result['turns']}")
print(f"Game log: {result['log_file']}")

In [ ]:
try:
    from google.colab import files
    files.download(result["log_file"])
except ImportError:
    print(f"Log file: {result['log_file']}")

---
## What's Next?

Once your agent completes this micro mission, move on to:
1. **Training Mission** (`the_hidden_layer_training_mission.ipynb`) — 5 dossiers, crafting, harder quests
2. **Full Mission** (`the_hidden_layer_full_mission.ipynb`) — 10 dossiers, full 8×8 complexity